### **BUILDING WHOLE GPT-2 MOE**

#### **IMPORTS NECESARIOS**

In [26]:
import os 
import sys
import torch
import urllib
import requests
import torch.nn as nn
from pathlib import Path


#### **1. INTEGRATING MOE IN TRANFORMER BLOCK**

##### **1.1 ATTENTION**

In [142]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in:int, d_out:int, context_length:int, attr:dict):

        # Initialiaze the parent
        super().__init__() 

        # Get att specific params
        num_heads:int = attr.get('num_heads', 6)
        dropout:float = attr.get('dropout', 0.1)
        qkv_bias:bool = attr.get('qkv_bias', False)

        # Get basic parameters
        self.d_out = d_out
        self.num_heads = num_heads
        # Force sum(dim(heads)) =  dout
        self.head_dim = d_out // num_heads 

        # Define the QKV matrices -> dk = dv = dout
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal = 1)   
        )

    def forward(self, x):
        # Get the shape (B, N, d_in)
        b, num_tokens, d_in = x.shape
        # Define the KQV matrices (B, num_tokens, d_out)
        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We modify shapes to consider all the heads
        # (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention 
        attn_scores = queries @ keys.transpose(2, 3)  

        # Original mask truncated to the number of tokens and converted to boolean
        # Addapted to the real amount of tokens in case input is not padded to obtain fixed N (context_length)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores -> upper diagonal MLM
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        # Get attention weights
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_heads, num_tokens, head_dim) -> (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where d_out -> num_heads * head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

##### **1.2 NORMALIZATION**

In [3]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim:int):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x:torch.Tensor)->torch.Tensor:
        """
            Scales data per instance and makes
            it follow a ~N(0, 1)
        """
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

##### **1.3 MOE BLOCK**

In [4]:
src_path = str(Path("/home/carlo/cuarto/GPT2_MOE/src").resolve())

# Asegurar que src tiene prioridad
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)

from gpt2_moe.model.moe import MOE

##### **1.4 TRANSFORMER BLOCK**

In [139]:
class TransformerBlock(nn.Module):
    def __init__(self, attr:dict):
        # Initialize parent
        super().__init__()
        # Common params
        self.d_in = attr["emb_dim"]
        self.d_out = attr["emb_dim"]
        self.cl = attr["context_length"]
        # Block specific params
        self.att_related = attr["attention_related"]
        self.moe_related = attr["moe_related"]
        
        # Define the attention module -> ok
        self.att = MultiHeadAttention(
           self.d_in,
           self.d_out,
           self.cl,
           self.att_related)

        # Define the ff layer
        self.ff = MOE(
                    self.d_in, 
                    self.d_out, 
                    self.moe_related) 

        # Define normalizations (within and previous)
        self.norm1 = LayerNorm(self.d_in)
        self.norm2 = LayerNorm(self.d_in)
        
        # Define the dropout
        self.drop_shortcut = nn.Dropout(attr['dropout'])

    def forward(self, x:torch.Tensor)->torch.Tensor:
        # Info message
        print('\nENTERING A TRANSFORMER BLOCK')
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        # (B, N, de) ->(B, N, de)
        x = self.att(x)  
        print(f'(TFBLOCK) Aspect of attention output ({x.shape})')
        x = self.drop_shortcut(x)
        # Add the original input back
        x = x + shortcut  

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        # Go through ff Layer
        x = self.ff(x)
        print(f'(TFBLOCK) Aspect of FF output ({x.shape})')
        x = self.drop_shortcut(x)
        # Add the original input back
        x = x + shortcut 

        print(f'(TFBLOCK) Aspect of block output ({x.shape})')
 

        return x

In [140]:
# Define config for the block
block_config = {
    "context_length": 1024, 
    "emb_dim": 768, 
    "dropout":0.1,     
    "attention_related":{
        "num_heads":8,
        "dropout":0.1,
        "qkv_bias":True
    },
    "moe_related":{
        "experts":3,
        "k_routing":1,
        "hidden_dim":768,
    }
}

In [141]:
# Define mock tensor -> currently embedded
mock = torch.randn(size = (10, 100, 768))

# Get the block
tf_block = TransformerBlock(block_config)

# Test the function
result = tf_block(mock)



Aspecto del buas True

ENTERING A TRANSFORMER BLOCK
(TFBLOCK) Aspect of attention output (torch.Size([10, 100, 768]))
(MOE) CALLING GATING MECHANISM
(router) Output of the linear:torch.Size([1000, 3])
(router) Output of the top_k:torch.Size([1000, 1])--(torch.Size([1000, 1]))
(router) Aspect of the -inf mask:torch.Size([1000, 3])
(router) BUILDING INDEXING MECHANISM
(router) ·Aspect of the first index:torch.Size([1000, 1])
(router) ·Aspect of the second index:torch.Size([1000, 1])
(router) ·Aspect of the later inf maskk:torch.Size([1000, 3])
(router) Aspect of the normalized weights:torch.Size([1000, 1])
(MOE) Aspect of the container for output tensor: torch.Size([1000, 768])
(MOE) EXPERT (0)
(expert) Output of the first linear:torch.Size([309, 768])
(expert) Output of the second linear:torch.Size([309, 768])
(expert) Aspect of the expert weights: torch.Size([309, 1])
(MOE) EXPERT (1)
(expert) Output of the first linear:torch.Size([347, 768])
(expert) Output of the second linear:torch.

#### **2. INTEGRATING MOE IN GPT BLOCK**


##### **2.1 GPT BLOCK**

In [143]:
class GPTMOE(nn.Module):
    def __init__(self, attr:dict):
        # Initialize parents
        super().__init__()
        
        # Define embedding layer
        self.tok_emb = nn.Embedding(attr["vocab_size"], attr["emb_dim"])
        self.pos_emb = nn.Embedding(attr["context_length"], attr["emb_dim"])
        self.drop_emb = nn.Dropout(attr["dropout"])

        # Build the transformer blocks
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(attr) for _ in range(attr["n_layers"])])


        # Output layer norm
        self.final_norm = LayerNorm(attr["emb_dim"])
        # Language head
        self.out_head = nn.Linear(
            attr["emb_dim"], attr["vocab_size"], bias=False
        )

    def forward(self, in_idx:torch.Tensor)->torch.Tensor:

        # Get input dims (B, N)
        batch_size, seq_len = in_idx.shape
        # Embed the inputs (B, N, de)
        # 1. Get contextual embeddings
        tok_embeds = self.tok_emb(in_idx)
        
        # 2. Get positional embeddings
        print(f'(GPT)CREATING EMBEEDINGS')
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        print(f'(GPT)Aspect of the embeddings:{x.shape}')

        # 3. Go trought transformer blocks (B, N, de) -> (B, N, de)
        x = self.trf_blocks(x)
        print(f'(GPT)Output of the trf blocks:{x.shape}')
        x = self.final_norm(x)

        # Get logits (B, N, de) ->(B, N, |V|)
        logits = self.out_head(x)
        print(f'(GPT)Aspect of the logits space:{x.shape}')

        return logits

##### **2.2 PARAM DEFINITION**

In [195]:
# Define the parameters
model_configs = {
    "vocab_size": 50257, 
    "context_length": 1024,
    "n_layers": 16,  
    "emb_dim": 768, 
    "dropout":0.1,     
    "attention_related":{
        "num_heads":8,
        "dropout":0.1,
        "qkv_bias":True
    },
    "moe_related":{
        "experts":3,
        "k_routing":1,
        "hidden_dim":768,
    },
        
    }

##### **2.3 TESTING**

In [254]:
# Create the gpt2 object
gpt2_moe = GPTMOE(model_configs)

# Define the mock tensor -> (B, N, dtype = int)
mock_tensor = torch.randint(low = 10, high = 100, size = (10, 100))
result = gpt2_moe(mock_tensor)


(GPT)CREATING EMBEEDINGS
(GPT)Aspect of the embeddings:torch.Size([10, 100, 768])

ENTERING A TRANSFORMER BLOCK
(TFBLOCK) Aspect of attention output (torch.Size([10, 100, 768]))
(MOE) CALLING GATING MECHANISM
(router) Output of the linear:torch.Size([1000, 3])
(router) Output of the top_k:torch.Size([1000, 1])--(torch.Size([1000, 1]))
(router) Aspect of the -inf mask:torch.Size([1000, 3])
(router) BUILDING INDEXING MECHANISM
(router) ·Aspect of the first index:torch.Size([1000, 1])
(router) ·Aspect of the second index:torch.Size([1000, 1])
(router) ·Aspect of the later inf maskk:torch.Size([1000, 3])
(router) Aspect of the normalized weights:torch.Size([1000, 1])
(MOE) Aspect of the container for output tensor: torch.Size([1000, 768])
(MOE) EXPERT (0)
(expert) Output of the first linear:torch.Size([352, 768])
(expert) Output of the second linear:torch.Size([352, 768])
(expert) Aspect of the expert weights: torch.Size([352, 1])
(MOE) EXPERT (1)
(expert) Output of the first linear:torch.

#### **3. WEIGHT LOADING**

##### **3.1 FILE DOWNLOAD**

In [ ]:
def download_gpt2(name: str, model_dir:str)->torch.Tensor:
    """  
        Function aimed to load GPT2 weights

        Params:
            name(str): 
                The name of the model
            model_dir(str):
                Base dir for model storage
            
        Returns:
            weights (torch.Tensor):
                Tensor representing weights
    
    """
    # Check path existance
    full_path = os.path.join(model_dir, name)
    if not os.path.exists(full_path):
        
        # Build the base folder 
        os.makedirs(model_dir, exist_ok = True)

        # Define the url 
        url = f"https://huggingface.co/rasbt/gpt2-from-scratch-pytorch/resolve/main/{file_name}"
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        with open(full_path, "wb") as f:
            f.write(response.content)
        print(f"Downloaded to {full_path}")

In [ ]:
# Download weights 
download_gpt2("gpt2-small-124M.pth", "weights")

##### **3.2 LOAD WEIGHTS**

In [196]:
# Get the state dict
state_dict = torch.load('weights/gpt2-small-124M.pth', map_location="cpu", weights_only=True)

In [197]:
def assign(left:torch.Tensor, right:torch.Tensor):
    """ 
        Function aimed to assign weights
    """
    if left.shape!=right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")

    # Assign weights
    return nn.Parameter(right)


In [241]:
def load_weights(model:GPTMOE, params:dict, blocks:int)->None:
    """
        Function aimed to load weights into a given 
        model. GPT skeleton architecture is expected

        Params:
            model (GPTMOE):
                The model to initialize


            params (dict):
                The dictionary containing weights

            blocks (int):
                The amount of tf blocks to initialize

        Returns:
            None:
                No value is returned

    """
    # Acquire embeddings 
    model.tok_emb.weight = assign(model.tok_emb.weight, params["tok_emb.weight"])
    model.pos_emb.weight = assign(model.pos_emb.weight, params["pos_emb.weight"])

    # Initialize tf blocks
    for i in range(blocks):
        # MHA Block
        # Initialize query
        model.trf_blocks[i].att.W_query.weight = assign(model.trf_blocks[i].att.W_query.weight,
                                                        params["trf_blocks.0.att.W_query.weight"])
        
        model.trf_blocks[i].att.W_query.bias = assign(model.trf_blocks[i].att.W_query.bias,
                                                            params["trf_blocks.0.att.W_query.bias"])
        # Initialize key 
        model.trf_blocks[i].att.W_key.weight = assign( model.trf_blocks[i].att.W_key.weight,
                                                      params["trf_blocks.0.att.W_key.weight"])
        
        model.trf_blocks[i].att.W_key.bias = assign( model.trf_blocks[i].att.W_key.bias,
                                                              params["trf_blocks.0.att.W_key.bias"])  
        # Initialize value
        model.trf_blocks[i].att.W_value.weight = assign( model.trf_blocks[i].att.W_value.weight,
                                                        params["trf_blocks.0.att.W_value.weight"]) 
        
        model.trf_blocks[i].att.W_value.bias = assign( model.trf_blocks[i].att.W_value.bias,
                                                            params["trf_blocks.0.att.W_value.bias"]) 
        # Initialize out proj
        model.trf_blocks[i].att.out_proj.weight = assign( model.trf_blocks[i].att.out_proj.weight, params["trf_blocks.0.att.out_proj.weight"])
        model.trf_blocks[i].att.out_proj.bias = assign( model.trf_blocks[i].att.out_proj.bias, params["trf_blocks.0.att.out_proj.bias"])


        # Initialize normalizations (layer norm 1 and 2)
        model.trf_blocks[i].norm1.scale = assign(model.trf_blocks[i].norm1.scale, params["trf_blocks.0.norm1.scale"])
        model.trf_blocks[i].norm1.shift = assign(model.trf_blocks[i].norm1.shift, params["trf_blocks.0.norm1.shift"])
        model.trf_blocks[i].norm2.scale = assign(model.trf_blocks[i].norm2.scale, params["trf_blocks.0.norm2.scale"])
        model.trf_blocks[i].norm2.shift = assign(model.trf_blocks[i].norm2.shift, params["trf_blocks.0.norm2.shift"])


##### **3.3 LOAD EXPERTS**

**Get next available name**

Will be needed when training each one and storing it

In [270]:
def get_next_name(path:str, key:str)->str:
    """  
        Function aimed to return the next available 
        name given a base namekey. Names will be 
        built using the format name_(next_integer)

        Params:
            path(str):
                The base path to look at

            key (str):
                The base name

        Returns:
            name(str):
                The first available name
    
    """

    # Get last valid idx
    last_idx = sorted([int(entry.name.split('.')[0][-1])for entry in os.scandir('names_test')])[-1] 
    print('(next_name) Aspect of the last used idx:', last_idx)
    # Create the new idx
    new_idx = last_idx + 1
    print('(next_name) Aspect of the new idx:', new_idx)
    # Build the new name
    new_name = key + '_' + str(new_idx)
    print('(next_name) Aspect of the new name:', new_name)
    return new_name

In [272]:
# Test the function
expert_name_to_store = get_next_name('names_test', 'name')

(next_name) Aspect of the last used idx: 2
(next_name) Aspect of the new idx: 3
(next_name) Aspect of the new name: name_3


**Expert loading**

Won't be accessed until all the experts are ready

In [ ]:
def load_experts(model:GPTMOE, count:int, base_path:str, block:int)->None:
    """  
        Function aimed to load the available experts 
        into the GPT2MOE architecture. This function won't 
        be accessed until the k experts have been individually
        trained. 

        Expected storing format for the experts is the following:
        experts/
            block_index/
                expert_0.pt
                expert_1.pt
                |
                |
                expert_n.pt

        Params:
            model(GPTMOE):
                The model whose experts we want to set up
            count(int):
                The amount of experts to use
            base_path (str):
                The base path where experts live
            block(int):
                The block where expert is being loaded
    """
    # Put together path and block
    path = os.path.join(base_path, str(block))
    print('(expert_load) Aspect of the path:', path)
    # Define expert chain
    name = 'expert_{index}.pth'
    # Iterate over the available experts 
    for i in range(count):
        print(f'(expert_load) Loading expert ({i})')
        # Get to the expert i 
        full_path = os.path.join(path, name.format(index = i))
        print(' ·Aspect of the full path:', full_path)
        # Retrieve the state dict of the expert
        expert_state_dict = torch.load(full_path)
        # Load the state dict into a selected model's module
        model.trf_blocks[block].ff.experts[i].load_state_dict(expert_state_dict)
    

In [329]:
# Create mock expert's state dicts
for i in range(1,3):
    for j in range(3):
        torch.save(gpt2_moe.trf_blocks[0].ff.experts[0].state_dict(), f'expert_load_test/{i}/expert_{j}.pth')


In [330]:
# Define the params
model = gpt2_moe
count = 3
base_path = 'expert_load_test'
block = 2

# Call the function
load_experts(model, count, base_path, block)


(expert_load) Aspect of the path: expert_load_test/2
(expert_load) Loading expert (0)
 ·Aspect of the full path: expert_load_test/2/expert_0.pth
(expert_load) Loading expert (1)
 ·Aspect of the full path: expert_load_test/2/expert_1.pth
(expert_load) Loading expert (2)
 ·Aspect of the full path: expert_load_test/2/expert_2.pth


/tmp/ipykernel_88324/1136778448.py:39: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  expert_state_dict = torch.load(full_path)
